In [1]:
import pandas as pd

path = "gs://northeastgroup4t-ml-bucket/base_tips_raw.csv"

df = pd.read_csv(path)

print("Rows:", len(df))
df.head()

Rows: 76


,sentence
0,Set attainable goals and reward yourself when ...
1,They can keep you accountable!
2,Tell your family and friends that you plan to ...
3,They can keep you accountable! Avoid harmful b...
4,Do things that make you happy


In [2]:
sentences = df['sentence'].tolist()

cleaned = []
i = 0
while i < len(sentences):
    s = str(sentences[i]).strip()

    # Case 1: sentence ends with '(ex' — complete the broken fragment
    if s.endswith('(ex'):
        cleaned.append(s + '. walking)')
        i += 1
        continue

    # Case 2: sentence starts with 'walking) ...' — strip the prefix, keep the remainder
    if s.startswith('walking) '):
        core = s[len('walking) '):].strip()
        if core:
            cleaned.append(core)
        i += 1
        continue

    # Case 3: isolated 'walking)' with no following content — discard
    if s == 'walking)':
        i += 1
        continue

    # Case 4: 'They can keep you accountable! ...' fused with next tip — keep remainder only
    if s.startswith('They can keep you accountable! '):
        core = s[len('They can keep you accountable! '):].strip()
        if core:
            cleaned.append(core)
        i += 1
        continue

    # Case 5: isolated 'They can keep you accountable!' — retain as standalone tip, strip punctuation
    if s == 'They can keep you accountable!':
        cleaned.append('They can keep you accountable')
        i += 1
        continue

    cleaned.append(s)
    i += 1

# Strip trailing punctuation (. or !)
cleaned = [s.rstrip('.!') for s in cleaned]

# Deduplicate while preserving original order
seen = set()
unique_tips = []
for tip in cleaned:
    if tip not in seen:
        seen.add(tip)
        unique_tips.append(tip)

base_tips_df = pd.DataFrame(unique_tips, columns=['sentence'])
print(f"Raw rows: {len(df)}")
print(f"After cleaning, before dedup: {len(cleaned)}")
print(f"Final base tips count: {len(base_tips_df)}")
print()
for idx, tip in enumerate(unique_tips, 1):
    print(f"{idx:3d}. {tip}")

Raw rows: 76
After cleaning, before dedup: 75
Final base tips count: 37

  1. Set attainable goals and reward yourself when you achieve them
  2. They can keep you accountable
  3. Tell your family and friends that you plan to quit smoking
  4. Avoid harmful behaviors, such as smoking and excessive drinking
  5. Do things that make you happy
  6. Avoid who stress you out: If someone is constantly causing stress in your life and you cannot break the relationship, limit the time you spend with it, or break the relationship completely if possible
  7. Get a journal and write your thoughts out
  8. When facing great challenges, try to look at them as opportunities for personal growth
  9. Anticipate and plan for the challenges that may arise
 10. Take a post-meal walk
 11. If you’re lacking certain nutrients, take supplements
 12. Share your feelings with a trusted friend in person or talk to a psychologist
 13. Change your mindset
 14. Drink plenty of water
 15. Read about the harmful eff

In [3]:
merge_map = {
    "Eat a balanced meal": "Eat a balanced meal. It should be composed of 25% proteins, 25% carbohydrates and 50% vegetables",
    "It should be composed of 25% proteins, 25% carbohydrates and 50% vegetables": "Eat a balanced meal. It should be composed of 25% proteins, 25% carbohydrates and 50% vegetables",
    "Change your mindset": "Change your mindset. Try to look at problems from a positive perspective. For example, when you are stuck in traffic, look at it as an opportunity to pause and listen to your favorite radio station",
    "Try to look at problems from a positive perspective": "Change your mindset. Try to look at problems from a positive perspective. For example, when you are stuck in traffic, look at it as an opportunity to pause and listen to your favorite radio station",
    "For example, when you are stuck in traffic, look at it as an opportunity to pause and listen to your favorite radio station": "Change your mindset. Try to look at problems from a positive perspective. For example, when you are stuck in traffic, look at it as an opportunity to pause and listen to your favorite radio station",
    "Tell your family and friends that you plan to quit smoking": "Tell your family and friends that you plan to quit smoking. They can keep you accountable",
    "They can keep you accountable": "Tell your family and friends that you plan to quit smoking. They can keep you accountable",
}

# Apply merge rules
merged = []
for tip in unique_tips:
    merged.append(merge_map.get(tip, tip))

# Deduplicate while preserving order
seen = set()
final_tips = []
for tip in merged:
    if tip not in seen:
        seen.add(tip)
        final_tips.append(tip)

base_tips_df = pd.DataFrame(final_tips, columns=['sentence'])
print(f"Before merge: {len(unique_tips)}")
print(f"After merge: {len(base_tips_df)}")
print()
for idx, tip in enumerate(final_tips, 1):
    print(f"{idx:3d}. {tip}")

Before merge: 37
After merge: 33

  1. Set attainable goals and reward yourself when you achieve them
  2. Tell your family and friends that you plan to quit smoking. They can keep you accountable
  3. Avoid harmful behaviors, such as smoking and excessive drinking
  4. Do things that make you happy
  5. Avoid who stress you out: If someone is constantly causing stress in your life and you cannot break the relationship, limit the time you spend with it, or break the relationship completely if possible
  6. Get a journal and write your thoughts out
  7. When facing great challenges, try to look at them as opportunities for personal growth
  8. Anticipate and plan for the challenges that may arise
  9. Take a post-meal walk
 10. If you’re lacking certain nutrients, take supplements
 11. Share your feelings with a trusted friend in person or talk to a psychologist
 12. Change your mindset. Try to look at problems from a positive perspective. For example, when you are stuck in traffic, loo

In [4]:
output_path = "gs://northeastgroup4t-ml-bucket/base_tips_final.csv"
base_tips_df.to_csv(output_path, index=False)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
